In [29]:
import numpy as np
import datasets
from datasets import load_dataset
import pandas as pd
import altair as alt
import numpy as np
from helper_funks import show_me_more
alt.renderers.enable("jupyter", offline=True)
alt.renderers.enable("mimetype")

RendererRegistry.enable('mimetype')

Initial data analysis

In [30]:
# Load our dataset
ds = load_dataset("trl-lib/tldr")
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 116722
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 6447
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 6553
    })
})

In [31]:
# Looking at the training, test and validation data to examine them
training = pd.read_excel(open('subsets.xlsx', 'rb'), sheet_name='Train')
testing = pd.read_excel(open('subsets.xlsx', 'rb'), sheet_name='Test')
validation = pd.read_excel(open('subsets.xlsx', 'rb'), sheet_name='Validation')

In [32]:
# showing the amount of posts in each subreddit (category) in the training set
bar = show_me_more(training, "Train data", 'bluepurple')
bar.display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [33]:
# showing the amount of posts in each subreddit (category) in the test set
bar = show_me_more(testing, "Test data", 'greenblue')
bar.display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [34]:
# showing the amount of posts in each subreddit (category) in the validation set
bar = show_me_more(validation, "Validation data", 'orangered', False)
bar.display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


Preliminary model evaluation and metrics analysis

In [9]:
#Import the TL;DR dataset.
from datasets import load_dataset
posts = load_dataset("trl-lib/tldr")

In [10]:
#Method for getting the subreddit from the prompt text.
def getSubredditFromPost(post):
    return  post['prompt'].split('\n')[0][11:];

#Get all subreddits and the amount of posts present within each subreddit.
counts = {}
subreddits = []
for post in posts['train']:
    subreddit = getSubredditFromPost(post)
    if subreddit not in subreddits:
        subreddits.append(subreddit)
    if(subreddit in counts):
        counts[subreddit] = counts[subreddit] + 1
    else:
        counts[subreddit] = 1

#To ensure equal representation of subreddits we evaluate the same amount of posts from each subreddit. 
#That means we will take the amount of posts in the subreddit that has fewest posts and evaluate this 
#amount of posts for each subreddit.
amount_of_posts_to_evaluate = min(counts.values())


In [11]:
import time
import evaluate
from helper_funks import NOIR

#Load the rouge and bleurt metric for evaluation.
rouge = evaluate.load("rouge")
bleurt = evaluate.load("bleurt", module_type="metric")

#Compute the NOIR, bleurt and rouge scores for the given pipeline.
def compute_scores(pipe, model_name):
  noirScores = {}
  bleurtScores = {}
  rougeScores = {}
  promtsAndSummaries = {}
  for subreddit in subreddits:
    start = time.time()

    noirScores[subreddit] = []

    #Get the prompts and the labels of the given subreddit.
    sub_dataset = posts.filter(lambda post : getSubredditFromPost(post) == subreddit)['train'].select(range(amount_of_posts_to_evaluate))
    sub_dataset_prompts = [p['prompt'] for p in sub_dataset]
    labels = [p['completion'] for p in sub_dataset]
    
    #Generate the summaries from the given pipeline.
    generatedTexts = [text['summary_text'] for text in pipe(sub_dataset_prompts)]
    #Compute the NOIR score for each summary
    for prompt, generatedText in zip(sub_dataset_prompts, generatedTexts):
      promtsAndSummaries["promt"] = prompt
      promtsAndSummaries[model_name] = generatedText
      
      noirScore = NOIR(prompt, generatedText)
      noirScores[subreddit].append(noirScore)
    
    #Compute the BLEURT score for each summary
    bleurtScore = bleurt.compute(predictions=generatedTexts, references=labels)
    bleurtScores[subreddit] = bleurtScore['scores']

    #Compute the average ROUGE-1, ROUGE-2, ROUGE-L and ROUGE-Lsum for the generated summaries.
    rougeScore = rouge.compute(predictions=generatedTexts, references=labels)
    rougeScores[subreddit] = rougeScore

    print(subreddit + " time: " + str(time.time()-start))

  return noirScores, bleurtScores, rougeScores, promtsAndSummaries


Using default checkpoint 'bleurt-base-128' for sequence maximum length 128. You can use a bigger model for better results with e.g.: evaluate.load('bleurt', config_name='bleurt-large-512').



INFO:tensorflow:Reading checkpoint C:\Users\Bruger\.cache\huggingface\metrics\bleurt\default\downloads\extracted\48268fdd2275c6770bfc2bae27f0b7350297e67459005b4af66dcadb1743f780\bleurt-base-128.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint bert_custom
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:bert_custom
INFO:tensorflow:... vocab_file:vocab.txt
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... do_lower_case:True
INFO:tensorflow:... max_seq_length:128
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating WordPiece tokenizer.

INFO:tensorflow:WordPiece tokenizer instantiated.
INFO:tensorflow:Creating Eager Mode predictor.
INFO:tensorflow:Loading model.
INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [ ]:
import pandas as pd
def storePromptsAndSummaries(promtsAndSummaries):
    original_df = pd.read_csv("./summaries.csv")
    new_df = pd.read_csv(promtsAndSummaries)
    final_df = pd.merge(original_df, new_df, on='prompt', how='outer')
    final_df.to_csv('./summaries.csv', index=False)
    

In [ ]:
#Evaluate the bart base model from ainize
from transformers import pipeline
ainize_noirScores, ainize_bleurtScores, ainize_rougeScores, promtsAndSummaries = compute_scores(pipeline("summarization", model="ainize/bart-base-cnn", device=0), "ainize/bart-base-cnn")
storePromptsAndSummaries(promtsAndSummaries)


In [ ]:
#Save the scores to a csv and a json file.
import pandas as pd
from transformers import pipeline
df = pd.DataFrame(ainize_noirScores)
df.to_json('ainize_noirScores.json', index=False)
df.to_csv('ainize_noirScores.csv', index=False)
df = pd.DataFrame(ainize_bleurtScores)
df.to_json('ainize_bleurtScores.json', index=False)
df.to_csv('ainize_bleurtScores.csv', index=False)
df = pd.DataFrame(ainize_rougeScores)
df.to_json('ainize_rougeScores.json', index=False)
df.to_csv('ainize_rougeScores.csv', index=False)

In [ ]:
#Evaluate the t5-small model from falconsai.
from transformers import pipeline
falconsai_noirScores, falconsai_bleurtScores, falconsai_rougeScores = compute_scores(pipeline("summarization", model="Falconsai/text_summarization", device=0))


In [ ]:
#Save the scores to a csv and a json file.
import pandas as pd
df = pd.DataFrame(falconsai_noirScores)
df.to_json('falconsai_noirScores.json', index=False)
df.to_csv('falconsai_noirScores.csv', index=False)
df = pd.DataFrame(falconsai_bleurtScores)
df.to_json('falconsai_bleurtScores.json', index=False)
df.to_csv('falconsai_bleurtScores.csv', index=False)
df = pd.DataFrame(falconsai_rougeScores)
df.to_json('falconsai_rougeScores.json', index=False)
df.to_csv('falconsai_rougeScores.csv', index=False)

In [ ]:
#Evaluate the pegasus large model from google
from transformers import pipeline
pipe = pipeline("summarization", model="google/pegasus-large", device=0)
pegasus_noirScores, pegasus_bleurtScores, pegases_rougeScores = compute_scores(pipe)

In [ ]:
#Save the scores to a csv and a json file.
import pandas as pd
df = pd.DataFrame(pegasus_noirScores)
df.to_json('pegasus_noirScores.json', index=False)
df.to_csv('pegasus_noirScores.csv', index=False)
df = pd.DataFrame(pegasus_bleurtScores)
df.to_json('pegasus_bleurtScores.json', index=False)
df.to_csv('pegasus_bleurtScores.csv', index=False)
df = pd.DataFrame(pegases_rougeScores)
df.to_json('pegases_rougeScores.json', index=False)
df.to_csv('pegases_rougeScores.csv', index=False)

Plotting the metrics

In [12]:
# Bart-Base: Ainize
# getting the data as dataframe and reformating them so we can plot them
df_ainize_bleurt = pd.read_csv('evals/ainize_bleurtScores.csv').melt(var_name='column')
df_ainize_noir = pd.read_csv('evals/ainize_noirScores.csv').melt(var_name='column')
# change names of the dataframes for easy plotting
source_1_1 = pd.DataFrame(df_ainize_bleurt)
source_1_2 = pd.DataFrame(df_ainize_noir)

In [13]:
# T5 small - falconsai
# getting the data as dataframe and reformating them so we can plot them
df_falconsai_bleurt = pd.read_csv('evals/falconsai_bleurtScores.csv').melt(var_name='column')
df_falconsai_noir = pd.read_csv('evals/falconsai_noirScores.csv').melt(var_name='column')
# change names of the dataframes for easy plotting
source_2_1 = pd.DataFrame(df_falconsai_bleurt)
source_2_2 = pd.DataFrame(df_falconsai_noir)

In [14]:
# Pegasus
# getting the data as dataframe and reformating them so we can plot them
df_pegasus_bleurt = pd.read_csv('evals/pegasus_bleurtScores.csv').melt(var_name='column')
df_pegasus_noir = pd.read_csv('evals/pegasus_noirScores.csv').melt(var_name='column')
# change names of the dataframes for easy plotting
source_3_1 = pd.DataFrame(df_pegasus_bleurt)
source_3_2 = pd.DataFrame(df_pegasus_noir)

In [15]:
# Plot of the three models with Bleurt metrics
one = alt.Chart(source_1_1, title="ainize_bleurt").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
two = alt.Chart(source_2_1, title="falconai_bleurt").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
three = alt.Chart(source_3_1, title="pegasus_bleurt").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)

(one & two & three).properties(width=800).interactive()
(one & two & three).display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [16]:
# Plot of the three models with Noir metrics
one = alt.Chart(source_1_2, title="ainize_noir").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
two = alt.Chart(source_2_2, title="falconai_noir").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
three = alt.Chart(source_3_2, title="pegasus_noir").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)

(one & two & three).properties(width=800).interactive()
(one & two & three).display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [17]:
# Rouge metrics
# Rouge metrics had a different format than Bleurt and Noir, so we needed to plot the results in a different way, we chose line charts
df_ainize_rouge = pd.read_csv('evals/ainize_rougeScores.csv')
df_falconsai_rouge = pd.read_csv('evals/falconsai_rougeScores.csv')
df_pegases_rouge = pd.read_csv('evals/pegases_rougeScores.csv')

# Melt the dataframes to long format
df_ain_rouge = df_ainize_rouge.reset_index().melt(id_vars='index', var_name='Subreddit', value_name='Value')
df_falc_rouge = df_falconsai_rouge.reset_index().melt(id_vars='index', var_name='Subreddit', value_name='Value')
df_peg_rouge = df_pegases_rouge.reset_index().melt(id_vars='index', var_name='Subreddit', value_name='Value')

In [18]:
# the line charts for the three models
one = (
    alt.Chart(df_ain_rouge, title='ainize_Rouge')
    .mark_line()
    .encode(
        x='Subreddit:N',
        y='Value:Q',
        color='index:N'
    )
)
two = (
    alt.Chart(df_falc_rouge, title='Falconai_Rouge')
    .mark_line()
    .encode(
        alt.X('Subreddit:N').axis(labels=False),
        y='Value:Q',
        color='index:N'
    )
)
three = (
    alt.Chart(df_peg_rouge, title='Pegasus_Rouge')
    .mark_line()
    .encode(
        alt.X('Subreddit:N').axis(labels=False),
        y='Value:Q',
        color='index:N'
    )
)

(one & two & three).properties(width=800).interactive()
(one & two & three).display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting
